# 2. Phân đoạn khách hàng với Hierarchical Clustering

**Mục tiêu:** Trực quan hóa cấu trúc phân cấp bằng Dendrogram, so sánh các phương pháp liên kết (`ward`, `complete`, `average`, `single`), chọn cấu hình phù hợp và lưu nhãn `hierarchical` vào `clustering/results/labels.csv`.

## Lý thuyết cốt lõi
Thuật toán phân cấp tích tụ (**Agglomerative Clustering** - Bottom-Up) bắt đầu với mỗi khách hàng là một cụm đơn lẻ, sau đó lặp lại việc gộp 2 cụm gần nhau nhất cho đến khi tất cả hợp thành một cây phân cấp (**Dendrogram**).

4 kiểu đo khoảng cách giữa 2 cụm (**linkage**):
- **`ward`:** Gộp 2 cụm sao cho tổng bình phương sai số nội cụm (SSE) tăng ít nhất (chỉ dùng với khoảng cách Euclidean; thường tạo các cụm tròn và cân đối nhất).
- **`complete`:** Khoảng cách lớn nhất giữa 2 điểm bất kỳ của 2 cụm (tránh kéo thành chuỗi dài).
- **`average`:** Trung bình khoảng cách của mọi cặp điểm giữa 2 cụm.
- **`single`:** Khoảng cách nhỏ nhất giữa 2 điểm của 2 cụm (nhạy nhiễu, dễ tạo hiệu ứng dây chuyền *chaining*).

## Quyết định lấy mẫu khi vẽ Dendrogram
Thuật toán phân cấp có độ phức tạp bộ nhớ $O(n^2)$ và thời gian $O(n^2 \log n)$–$O(n^3)$. Nếu vẽ Dendrogram cho toàn bộ khách hàng thì biểu đồ sẽ quá dày và rất chậm; vì vậy ta **lấy mẫu ngẫu nhiên 2.000 khách hàng** (`random_state=42`) để vẽ cây, còn khi chạy `AgglomerativeClustering` lấy nhãn cuối vẫn chạy trên **toàn bộ dữ liệu** để bàn giao đủ cho TV3.

In [ ]:
import sys
from pathlib import Path

# Thêm thư mục gốc của repo vào sys.path để import được clustering/models.py
# dù bạn mở notebook từ thư mục gốc hay từ thư mục clustering/
ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "clustering").exists() and (ROOT_DIR.parent / "clustering").exists():
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import matplotlib.pyplot as plt
import pandas as pd
import scipy.cluster.hierarchy as sch
from sklearn.cluster import AgglomerativeClustering

from clustering.models import (
    RANDOM_STATE,
    fit_hierarchical,
    load_scaled_data,
    run_grid_search,
    save_cluster_labels,
    save_tuning_results,
)

customer_ids, X_df, is_mock = load_scaled_data()
print(f"Kích thước dữ liệu đưa vào mô hình: {X_df.shape}")

## Bước 1: Vẽ Dendrogram trên mẫu 2.000 khách hàng
- Dùng `truncate_mode='lastp', p=30` để chỉ hiển thị 30 nhánh gộp cuối cùng ở đỉnh cây cho dễ quan sát.
- **Cách đọc:** Tìm khoảng cách dọc (trục tung) dài nhất trước khi các nhánh bị gộp lại; tưởng tượng một đường cắt ngang đi qua khoảng đó — số nhánh dọc bị cắt chính là số cụm tự nhiên.

In [ ]:
SAMPLE_SIZE = min(2000, len(X_df))
X_sample = X_df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE)

LINKAGE_METHODS = ["ward", "complete", "average", "single"]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, method in zip(axes.ravel(), LINKAGE_METHODS):
    # Tính ma trận liên kết bằng SciPy (metric mặc định là euclidean)
    Z = sch.linkage(X_sample.to_numpy(), method=method)
    sch.dendrogram(
        Z,
        ax=ax,
        truncate_mode="lastp",
        p=30,
        no_labels=True,
        color_threshold=None,
    )
    ax.set_title(f"Dendrogram - linkage = '{method}' (n={SAMPLE_SIZE})")
    ax.set_xlabel("Cụm con được gộp (rút gọn 30 nhánh cuối)")
    ax.set_ylabel("Khoảng cách gộp")

plt.tight_layout()
plt.show()

## Bước 2: Tinh chỉnh `linkage` và `n_clusters` trên toàn bộ dữ liệu
Quét lưới 4 kiểu `linkage` và `n_clusters` từ `2` đến `8` để so sánh định lượng qua **Silhouette**, **Davies-Bouldin**, và **Calinski-Harabasz** (cố định `metric='euclidean'` để so sánh công bằng vì `ward` bắt buộc dùng Euclidean).

In [ ]:
K_RANGE = list(range(2, 9))

tuning_df = run_grid_search(
    X_df,
    model_builder=lambda linkage, n_clusters: AgglomerativeClustering(
        n_clusters=n_clusters,
        linkage=linkage,
        metric="euclidean",
    ),
    param_grid={
        "linkage": LINKAGE_METHODS,
        "n_clusters": K_RANGE,
    },
)

save_tuning_results(tuning_df, "tuning_hierarchical.csv")
tuning_df.sort_values(by="silhouette", ascending=False).head(10)

## Bước 3: Biểu đồ so sánh các kiểu `linkage`
Quan sát đường Silhouette ($\uparrow$) và Davies-Bouldin ($\downarrow$) giúp nhận ra ngay kiểu linkage nào phân cụm cân đối và tách biệt rõ nhất (thường `ward` cho kết quả tốt hơn `single` trên dữ liệu khách hàng).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for method in LINKAGE_METHODS:
    sub = tuning_df[tuning_df["linkage"] == method].sort_values("n_clusters")
    axes[0].plot(sub["n_clusters"], sub["silhouette"], marker="o", label=method)
    axes[1].plot(sub["n_clusters"], sub["davies_bouldin"], marker="s", label=method)

axes[0].set_title("Silhouette Score theo linkage & số cụm")
axes[0].set_xlabel("Số cụm (k)")
axes[0].set_ylabel("Silhouette (càng cao càng tốt)")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.5)

axes[1].set_title("Davies-Bouldin Index theo linkage & số cụm")
axes[1].set_xlabel("Số cụm (k)")
axes[1].set_ylabel("Davies-Bouldin (càng thấp càng tốt)")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## Bước 4: Chốt tham số cuối & Lưu nhãn bàn giao
> **Ghi chú cho báo cáo Chương 4 (cập nhật sau khi chạy trên data thật):**
> - Chọn `BEST_LINKAGE = ...` và `BEST_K = ...` dựa trên vết cắt Dendrogram kết hợp chỉ số Silhouette / Davies-Bouldin ở bảng trên.

In [ ]:
# >>> CHỈNH THAM SỐ CUỐI CÙNG SAU KHI XEM DỮ LIỆU THẬT <<<
BEST_LINKAGE = "ward"
BEST_K = 4

final_hc, hc_labels = fit_hierarchical(
    X_df,
    n_clusters=BEST_K,
    linkage=BEST_LINKAGE,
)

counts = pd.Series(hc_labels, name="count").value_counts().sort_index()
print("Phân bố số khách hàng theo cụm (Hierarchical):")
print(counts)

save_cluster_labels(customer_ids, "hierarchical", hc_labels)